# Deriving all needed dictionaries from Schematics and Perimeters

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import geopandas as gpd
import pandas as pd
import os
import glob
import pathlib as pl
import re
import json
import ast
import numpy as np
import datetime

## Inputs Required

In [3]:
#manual input - must provide the path of the home where the notebook folder holding this jupyter notebook will live in addition to the functions required.
os.chdir('..')
os.chdir('..')

home = pl.Path(os.getcwd())
print(home)

#location of all larger functions required for the perimeter delineation
from src.perimeter_functions_edited import *
#Location of all manually defined variables for the code
from _pyfile_notebook_edits.schematics_to_dictionaries_inputs import *

c:\_code\hms_to_ras_sst


__The following inputs are required for this code:__

>- Geodatabases/Geopackages for all HUC10s with a uniform name between them
>- Shapefile of the Subbasins and their HUC10 assignments from when perimeters were finalized for hydraulic models. 
>    - These assignments will be compared to the new schematics at some point of the code in case of name changing between initial and final schematic versions.
>- Shapefile of the final Downstream Junctions assigned after perimeters have been finalized.
>    - These will also need to be compared with the initial schematics in a quick overview to ensure that the locations are as expected relative to the HUC10 perimeters.
>- Shapefile of the Final perimeters as used for the models. For the input, there should be NO GAPS, and NO OVERLAPS between HUC10s

__Place these within the defined Inputs folder__

In [4]:
#consider adding a portion with the function that grabs and unites all features from the geodatabases identified in the inputs. Naming should be consistent between geodatabases.
# print(str(inputs)+schematic_input_type)
huc8_gdb = glob.glob(str(inputs)+schematic_input_type, recursive=False)
print('Identified inputs:',huc8_gdb)

junctions = locate_string_gdb_to_concat_gdf(huc8_gdb,"Junction")
reaches = locate_string_gdb_to_concat_gdf(huc8_gdb,"Reach")
sinks = locate_string_gdb_to_concat_gdf(huc8_gdb,"Sink")
reservoirs = locate_string_gdb_to_concat_gdf(huc8_gdb,"Reservoir")
sources = locate_string_gdb_to_concat_gdf(huc8_gdb,"Source")
subbasins = locate_string_gdb_to_concat_gdf(huc8_gdb,"Subbasin")

print(f"There are {len(junctions)} Junctions, {len(reaches)} Reaches, {len(sinks)} Sinks, {len(reservoirs)} Reservoirs, and {len(sources)} Sources.") 

assert junctions.crs == sinks.crs == reaches.crs == reservoirs.crs == sources.crs == subbasins.crs, 'ERROR.The geodataframes created are not the same projection. please review before continuing.'

#troubleshoot for prior error that has occurred. Ensuring reservoirs geometry field is set as the geometry
reservoirs = reservoirs.to_crs(sinks.crs).set_geometry('geometry')

#This shapefile is different from the raw schematic file to be received. It has the assigned HUC10s in a field. This is an output of the perimeter code and should be kept updated in early stages of perimeter modification.
#For example, if a downstream junction is changed, ensure all subbasins in this file are kept updated to reflect these changes. All subbasins should have an assigned HUC10 in the field.                   
old_subbasins = gpd.read_file(sub_path)                                                        
#This should be the same as the code output, or should be updated to reflect latest changes. Small perimeter changes do not impact the ds junction - only if you are reassigning entire subbasins to a different HUC10              
ds_junc = gpd.read_file(dsj_path)       
#final perimeters but prior to major changes - no overlaps or gaps between or within the HUC10 boundaries
huc10s_gdf = gpd.read_file(perim_path)

print('Read all present files')

c:\_code\hms_to_ras_sst\inputs\wy_fy23\schematics/*.gdb
Identified inputs: ['c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080001_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080002_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080003_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080004_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080005_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080006_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080007_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080008_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080009_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080010_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\\schematics\\HUC_10080011_SST.gdb', 'c:\\_code\\hms_to_ras_sst\\inputs\\wy_fy23\

c:\Users\magomez\.conda\envs\regularhome\lib\site-packages\geopandas\array.py:1406: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as USA_Contiguous_Albers_Equal_Area_Conic_USGS_version (the single non-null crs provided).
  warnings.warn(


After searching through 15 geodatabases/geopackages, there are now 1 "Reservoir" features
After searching through 15 geodatabases/geopackages, there are now 18 "Source" features
After searching through 15 geodatabases/geopackages, there are now 6965 "Subbasin" features
There are 4378 Junctions, 4378 Reaches, 16 Sinks, 1 Reservoirs, and 18 Sources.
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "fiona\ogrext.pyx", line 136, in fiona.ogrext.gdal_open_vector
  File "fiona\_err.pyx", line 291, in fiona._err.exc_wrap_pointer
fiona._err.CPLE_OpenFailedError: c:\_code\hms_to_ras_sst\inputs\wy_fy23\schematics\bighorn_subbasins_merged_250721.shp: No such file or directory

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\magomez\.conda\envs\regularhome\lib\site-packages\IPython\core\interactiveshell.py", line 3460, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\magomez\AppData\Local\Temp\ipykernel_28432\2142250185.py", line 22, in <module>
    old_subbasins = gpd.read_file(sub_path)
  File "c:\Users\magomez\.conda\envs\regularhome\lib\site-packages\geopandas\io\file.py", line 259, in _read_file
    return _read_file_fiona(
  File "c:\Users\magomez\.conda\envs\regularhome\lib\site-packages\geopandas\io\file.py", line 303, in _read_file_f

In [5]:
#merged gdf will be exported as shapefiles for reference while running code to check and troubleshoot
sinks.to_file(working_outputs/"sinks_trial_REVIEW.shp")
junctions.to_file(working_outputs/"junctions_trial_REVIEW.shp")
reaches.to_file(working_outputs/"reaches_trial.shp")
reservoirs.to_file(working_outputs/"reservoirs_trial.shp")
sources.to_file(working_outputs/"sources_trial.shp")

C:\Users\magomez\AppData\Local\Temp\ipykernel_40620\708822639.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  sinks.to_file(working_outputs/"sinks_trial_REVIEW.shp")
C:\Users\magomez\AppData\Local\Temp\ipykernel_40620\708822639.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  junctions.to_file(working_outputs/"junctions_trial_REVIEW.shp")
C:\Users\magomez\AppData\Local\Temp\ipykernel_40620\708822639.py:4: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  reaches.to_file(working_outputs/"reaches_trial.shp")
C:\Users\magomez\AppData\Local\Temp\ipykernel_40620\708822639.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  reservoirs.to_file(working_outputs/"reservoirs_trial.shp")
C:\Users\magomez\AppData\Local\Temp\ipykernel_40620\708822639.py:6: UserWarning: Column 

In [6]:
#join the tables for the latest schematics of subbasins and double check that their assignments are the same as expected. This join is done with the assumption subbasin names have not changed and will require manual checking
sub_huc_assigned = old_subbasins.set_index('name')[subb_field].to_dict()
subbasins[subb_field] = subbasins['name'].map(sub_huc_assigned)

subbasins.to_file(working_outputs/"subbasins_temp_assigned_REVIEW.shp")

C:\Users\magomez\AppData\Local\Temp\ipykernel_40620\3888537860.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  subbasins.to_file(working_outputs/"subbasins_temp_assigned_REVIEW.shp")


In [7]:
#quick check to see if subbasin centroids fall within the 
subb_centroid_series = subbasins.geometry.representative_point()
subbasin_centroids = gpd.GeoDataFrame(subbasins.drop(columns=['geometry']), geometry=subb_centroid_series, crs=subbasins.crs)
dissolved_subbasins_old = old_subbasins.dissolve(by=subb_field)
dissolved_subbasins_old_reset = dissolved_subbasins_old.reset_index()

new_to_old_sub_assign = gpd.sjoin(subbasin_centroids, dissolved_subbasins_old_reset)

discrepancy_subb_assignments = new_to_old_sub_assign.loc[new_to_old_sub_assign['huc_mod_left'] != new_to_old_sub_assign['huc_mod_right']]['name_left'].to_list()

if len(discrepancy_subb_assignments) == 0:
    print(f'No notable issues based on preliminary check. Review subbasin assigned shapefile in {working_outputs} for final check')
else:
    print(f'Review subbasin assigned shapefile in {working_outputs}. list of discrepancies has identified {discrepancy_subb_assignments}')

No notable issues based on preliminary check. Review subbasin assigned shapefile in C:\_code\outputs\wy22_dictionary_updates\working_outputs_dictionary for final check


In [8]:
#create a temporary dictionary assigning sinks to their nearest junction. This is being created new due to potential changes since initial schematic usage
sinks_list = sinks['name'].to_list()
sinks_to_junction = dict()

print('Prepare to type the nearest junction downstream of the sink without any quotes or spaces around it. Ex. HUC_000_J_123.\nIf the sink is the most downstream, type OUT. If it is closed basin type N\A')
for sink in sinks_list:
    user_input_connection = input(f'Please review sink {sink}.')
    sinks_to_junction[sink] = user_input_connection
    leaving_system = re.search(r'(?i)out', user_input_connection)
    
    if leaving_system:
        text = 'The sink is the most downstream'
        print(f"\033[31m{text}\033[0m")
        
with open(working_outputs/'sinks_to_junction.json','w') as outfile:
    json.dump(sinks_to_junction,outfile, indent=2)

Please review sink HUC_001_Sink_0. Type the nearest junction downstream of the sink without any quotes or spaces around it. Ex. HUC_000_J_123.
If the sink is the most downstream, type OUT. If it is closed basin type N\A 'HUC_005_J_336'
Please review sink HUC_002_Sink_0. Type the nearest junction downstream of the sink without any quotes or spaces around it. Ex. HUC_000_J_123.
If the sink is the most downstream, type OUT. If it is closed basin type N\A 'HUC_001_J_519'
Please review sink HUC_003_Sink_0. Type the nearest junction downstream of the sink without any quotes or spaces around it. Ex. HUC_000_J_123.
If the sink is the most downstream, type OUT. If it is closed basin type N\A 'HUC_002_J_228'
Please review sink HUC_004_Sink_0. Type the nearest junction downstream of the sink without any quotes or spaces around it. Ex. HUC_000_J_123.
If the sink is the most downstream, type OUT. If it is closed basin type N\A 'HUC_005_J_336'
Please review sink HUC_005_Sink_0. Type the nearest junc

In [11]:
sinks_to_junction

{'HUC_001_Sink_0': 'HUC_005_J_336',
 'HUC_002_Sink_0': 'HUC_001_J_519',
 'HUC_003_Sink_0': 'HUC_002_J_228',
 'HUC_004_Sink_0': 'HUC_005_J_336',
 'HUC_005_Sink_0': 'HUC_007_J_100',
 'HUC_005_Sink_1': 'HUC_005_J_302',
 'HUC_006_Sink_0': 'HUC_005_J_237',
 'HUC_007_Sink_0': 'HUC_010_J_350',
 'HUC_008_Sink_0': 'HUC_007_J_582',
 'HUC_009_Sink_0': 'HUC_010_J_350',
 'HUC_010_Sink_0': 'OUT',
 'HUC_011_Sink_0': 'HUC_010_J_115',
 'HUC_012_Sink_0': 'HUC_014_J_359',
 'HUC_013_Sink_0': 'HUC_012_J_139',
 'HUC_014_Sink_0': 'HUC_010_J_201',
 'HUC_016_Sink_0': 'OUT'}

## Creating the main dictionaries related to each HUC10

In [13]:
#Create a set from the perimeters to identify the unique HUC10s 
huc10_set = set(subbasins[subb_field].to_list())

In [14]:
#combining junctions, reservoirs and sinks into one geodataframe since subbsins may connect to any of them

#concatenate the sinks junctions and reservoirs into one geodataframe
junc_res_sink = pd.concat([reservoirs,junctions,sinks], ignore_index=True)

### First Group of Dictionaries

In [17]:
#Dictionaries that will be filled out
HUC10_Junc_Sub = {}         #HUC10 to Junctions belonging to HUC10 model to Subbasins in model
HUC10_Junc = {}             #HUC10 to Junctions belonging to HUC10 model
HUC10_Sub = {}              #HUC10 to Subbasins in model
Junc_Sub = {}               #Junctions and the Subbasins directly flowing into it (incremental)

In [18]:
total_subbasins = set()
total_junc_res_sink = set()

#identify all the junctions and subbasins belonging to each HUC10
for huc10 in huc10_set:
    group_subb_gdf = subbasins.loc[subbasins[subb_field] == huc10]
    total_subbasins.update(group_subb_gdf['name'].to_list())
    total_junc_res_sink.update(group_subb_gdf['Downstream'].to_list())
    
    group_Junc_Sub = dict()
    group_Sub_Junc = group_subb_gdf.set_index('name')['Downstream'].to_dict()
    for v in group_Sub_Junc.values():
        temp_subbasin_list = [i for i,j in group_Sub_Junc.items() if j == v]
        group_Junc_Sub[v] = temp_subbasin_list 
    
    HUC10_Junc_Sub[huc10] = group_Junc_Sub
    HUC10_Junc[huc10] = list(group_Junc_Sub.keys())
    HUC10_Sub[huc10] = group_subb_gdf['name'].to_list()
    Junc_Sub.update(group_Junc_Sub)

#identify any missing Junctions or subbasins in the final dictionaries. None should be missing. If so, they will need to be added in spatially to the dictionaries.
#confirm with curtis if they are needed if none of the subbasins directly flow into them as according to the subbasin information
missing_sub = [x for x in subbasins['name'].to_list() if x not in total_subbasins]
missing_j = [x for x  in junc_res_sink['name'].to_list() if x not in total_junc_res_sink]

assert len(missing_sub) == 0, 'Subbasin(s) are missing from the dictionaries. Must review cause. Could be missing assignment'

if len(missing_j) != 0:
    print('Junctions reservoirs or sinks are missing from the dictionaries. Must review cause. Could be lack of connectivity to Subbasins')

Junctions reservoirs or sinks are missing from the dictionaries. Must review cause. Could be lack of connectivity to Subbasins


In [43]:
a = 0
for j in Junc_Sub.values():
    a+=len(j)
assert len(subbasins)==a, 'The number of subbasins in the dictionary does not capture the total number of subbasins actually present in the schematic files.'

In [19]:
#If there are any features that are lacking connectivity in relation to the Subbasins, they will be listed within the missing_j list
missing_j

['Anchor_Reservoir',
 'HUC_001_J_519',
 'HUC_005_J_336',
 'HUC_010_J_350',
 'HUC_013_J_35',
 'HUC_013_J_135',
 'HUC_014_J_359',
 'HUC_005_Sink_1']

__Current acceptable reasons for Junctions (original layer type) missing connections to subbasins__

>- They are downstream from a Sink

In [20]:
#quick check if reasonable for their disconnectivity to subbasins
not_downstream_of_sink = [x for x in missing_j if x not in sinks_to_junction.values()]
print('Likely acceptable reason for junction disconnectivity.') if len(not_downstream_of_sink) == 0 else print(f'Currently {not_downstream_of_sink} are not downstream of a junction. Review whether acceptable. If so - continue')

Currently ['Anchor_Reservoir', 'HUC_013_J_35', 'HUC_013_J_135', 'HUC_005_Sink_1'] are not downstream of a junction. Review if acceptable. If so - continue


In [21]:
#if there are any missing subbasins of junctions in the dictionaries then they will need to be determined using a spatial intersection of the data with the perimeters provided.
if len(missing_j) != 0:
    sub_check = subbasins.loc[subbasins['Downstream'].isin(missing_j)]
    if len(sub_check) == 0:
        print('Features lack connection to Subbasins. They will be added to the dictionary spatially based on Perimeter shapefile.')
        missing_junc_res_sink_gdf = junc_res_sink.loc[junc_res_sink['name'].isin(missing_j)]
        joined_sub_hucgdf = gpd.sjoin(missing_junc_res_sink_gdf, huc10s_gdf, how='left', predicate='intersects')
        
        temp_junc_huc = joined_sub_hucgdf.set_index('name').to_dict()[perim_field]
        total_junc_res_sink.update(temp_junc_huc.keys())
        
        for i,j in temp_junc_huc.items():
            temp_junc_sub =  HUC10_Junc_Sub[j]
            temp_junc_sub.update({i:''})
            HUC10_Junc_Sub[j] = temp_junc_sub
            HUC10_Junc[j] = list(temp_junc_sub.keys())
            Junc_Sub[i] = ''
    else:
        print('Missing junctions connect to subbasins. Revise why not included in dictionaries')
else:
    print('No troubleshooting required for junctions.')

Features lack connection to Subbasins. They will be added to the dictionary spatially based on Perimeter Shapefile.


### Second Group of Dictionaries

In [22]:
#Dictionaries that will be filled out
HUC10_to_ds_junc = {}                 # HUC10 and their downstream junction 
ds_junc_to_HUC10 = {}                 # Downstream Junction and the HUC10 that they flow *INTO
HUC10_dsjunc_HUC10 = {}               #Nested dictionary for HUC10 to Junc to HUC10
HUC10_to_HUC10 = {}                   # HUC10 to what HUC10 they flow into
HUC10_to_HUC10_outofscope = {}        # HUC10s out of scope and what HUC10 they flow into

In [23]:
#combine the sinks junctions reservoirs and sources. 
si_so_re_ju = pd.concat([sinks,junctions,reservoirs,sources], ignore_index=True)

In [24]:
#create dictionaries that show their connection in network. Using this it'll be possible to identify what HUC goes to which HUC10 and create a dictionary for it. Features between HUC8s are now part of the same network
node_to = si_so_re_ju.set_index('name').to_dict()['Downstream']
node_to.update(sinks_to_junction)
reach_to = reaches.set_index('name').to_dict()['Downstream']

In [25]:
#Create dictionary with information across all layer provided. Below will create a list ignoring dataframes that are empty. 
initial_list = [junctions,reaches,reservoirs,sources]
edited_list = [i for i in initial_list if len(i) != 0]

connection_dictionary = single_dict_creator(gdf_list=edited_list,identify_field='name',connection_field='Downstream',supplemental_dicts=[sinks_to_junction])
#identify values that are not found in any of the keys provided

#the connection dictionary summarizing the one way connection among all provided dataframes must have values present in the keys. If there is a value not in the keys, it means that the feature is not present in any of the layers. must review.
unidentified_name_origin = [v for k,v in connection_dictionary.items() if v not in connection_dictionary.keys() and v != 'OUT']
assert len(unidentified_name_origin) == 0, 'Unidentified values in connection dictionary. They do not have features with matching names among Junctions reaches, reservoirs, sinks and sources'

In [26]:
#prior to filling out the dictionaries, provide a list of all the perimeters that are NOT scoped
userinput2 = input("Provide in list format the HUC10s that have perimeters/downstream junctions but are NOT SCOPED for hydraulic modelling. These will be placed in a separate dictionary from scoped models. \nEx. you can type [\'1000000001\',\'1000000002\']")
not_scoped = ast.literal_eval(userinput2)

#filling out the dictionaries above as needed 
HUC10_to_ds_junc = ds_junc.set_index(dsj_field).to_dict()['name']
ds_junc_to_HUC10 = ds_junc.set_index('name').to_dict()[dsj_field]

#check that the previous downstream junctions/sinks are actually named the same.
ds_junc_not_found = {k:v for k,v in HUC10_to_ds_junc.items() if v not in junc_res_sink['name'].to_list()}

if len(ds_junc_not_found.keys()) != 0:
    print(f'Downstream junctions is not part of junctions reservoirs or sinks. See: {ds_junc_not_found} \nThis will require a manual override.')
    
    huc_dsjunc_mend = dict()
    dsjunc_huc_mend = dict()
    for i,j in ds_junc_not_found.items():
        userinput_fixds = input(f'Please type the actual ds_junction name for HUC10 {i}. Type name alone with no apostrophe. Ex: HUC_001_Sink0')
        fixds = userinput_fixds
        
        huc_dsjunc_mend[i] = fixds
        dsjunc_huc_mend[fixds] = i
        del ds_junc_to_HUC10[j]
        

    #amend dictionaries with ds_junction information.
    HUC10_to_ds_junc.update(huc_dsjunc_mend)
    ds_junc_to_HUC10.update(dsjunc_huc_mend)

#identify the nodes that simply lack a connection at all. the nan value must be altered
nan_connection_dictionary = {k:v for k,v in connection_dictionary.items() if v is np.nan}
for k in nan_connection_dictionary.keys():
    connection_dictionary[k] = 'missing connection'

# ##############
# print(ds_junc_to_HUC10)
# #########

#now iterate through each downstream junction to fill in the needed dictionaries
for dsj, huc10 in ds_junc_to_HUC10.items():
    firstj_list = []
    firstj_list.append(dsj)

    # ######
    # print('downstream junction',firstj_list)
    
    nextj = singular_network(firstj_list,connection_dictionary,restriction_str='_R_')
    
    # ######
    # print('first iteration', nextj)
    
    sink_present = [x for x in nextj if x in sinks['name'].to_list()]
    
    # ######
    # print('if sinks present',sink_present)
    
    while len(sink_present) > 0:
        nextj = singular_network(nextj,connection_dictionary,restriction_str='_R_')
        sink_present = [x for x in nextj if x in sinks['name'].to_list()]
    
    #temporary value set until identified
    nexthuc = ''
    if nextj == ['OUT']:
        print(f'{huc10} has no next junction, next HUC10 will be set to \'OUT\'')
        nexthuc = 'OUT'
    if nextj == ['N\A']:
        print(f'{huc10} has no next junction, it is closed. next HUC10 will be set to \'N\A\'')
        nexthuc = 'N\A'
    
    for i,j in HUC10_Junc.items():        
        if nextj[0] in list(j):
            nexthuc = i
    
    HUC10_to_ds_junc[huc10] = dsj #these are also defined earlier in code -> if troubleshooting
    ds_junc_to_HUC10[dsj] = nexthuc #these are also defined earlier in code -> if troubleshooting
    HUC10_dsjunc_HUC10[huc10] = {dsj:nexthuc}
    
    if huc10 not in not_scoped:
        HUC10_to_HUC10[huc10] = nexthuc
    else:
        HUC10_to_HUC10_outofscope[huc10] = nexthuc
    
print('identified order of the HUC10s for dictionary. Please review.')

Provide in list format the HUC10s that have perimeters/downstream junctions but are NOT SCOPED for hydraulic modelling. These will be placed in a separate dictionary from scoped models. 
Ex. you can type ['1000000001','1000000002'] ["1008001004","1008001005","1008001008","1008001007","1008001602","1008001601","1008001603","1008001604","1008001009","1008001605"]
Downstream junctions is not part of junctions reservoirs or sinks. See: {'1008001605': 'Sink_10080016'} 
This will require a manual override.
Please type the actual ds_junction name for HUC10 1008001605. Type name alone with no apostrophe. Ex: HUC_001_Sink0 HUC_016_Sink_0
1008001009 has no next junction, next HUC10 will be set to 'OUT'
1008001605 has no next junction, next HUC10 will be set to 'OUT'
identified order of the HUC10s for dictionary. Please review.


In [27]:
#Quick check, may need review. This will print all connection dictionary entries in which the value does not have the expected naming convention of a typical junction or reach.
notin = {k:v for k,v in connection_dictionary.items() if 'HUC' not in v}

In [28]:
notin

{'HUC_007_J_16': 'Anchor_Reservoir',
 'HUC_010_Sink_0': 'OUT',
 'HUC_016_Sink_0': 'OUT'}

In [29]:
len(HUC10_to_HUC10)

68

In [30]:
#add an edit for the dictionaries of the HUC10 connections if they do not connect as expected given the schematic connections. For example, of a model is actually closed basin, Put 'N\A'
userinput3 = input('If there is any HUC10_to_HUC10 overrides that are desired, please input in dictionary format.\nNote: changes are uncommon unless models have shown deviation from expected flow such as being closed basin when not expected (TYPE \'N\A\' as value), etc.\n\nIf no changes, provide empty dictionary. Type {}. \nExample of what to type if change required- {\'1234567890\':\'N\A\',\'0987654321\':\'OUT\'}')
updated_connection = ast.literal_eval(userinput3)

for key,val in updated_connection.items():
    if key in not_scoped:
        HUC10_to_HUC10_outofscope.update(updated_connections)
    else:
        HUC10_to_HUC10.update(updated_connections)
    dsj_temp = HUC10_to_ds_junc[key]
    ds_junc_to_HUC10[dsj_temp] = val
    HUC10_dsjunc_HUC10[key] = {dsj_temp:val}             

If there is any HUC10_to_HUC10 overrides that are desired, please input in dictionary format.
Note: changes are uncommon unless models have shown deviation from expected flow such as being closed basin when not expected (TYPE 'N\A' as value), etc.

If no changes, provide empty dictionary. Type {}. 
Example of what to type if change required- {'1234567890':'N\A','0987654321':'OUT'} {}


### Third group of dictionaries

In [ ]:
#Dictionaries relating Sources to downstream and to nearest related sink
source_ds_to = {}                       #sources downstream to next junction
source_if_res = {}                      #source if significant reservoir - unique hydraulic modeling applies
source_if_inc = {}                      #source if incremental - flow is added to junction it ties into rather than replacing it

hms_origin_dictionary = {}              #hms origin of every feature

rtoj = {}                               #reaches to any node (junctions sinks reservoirs sources)
jtor = {}                               #(sinks reservoirs sources) junctions to reaches
rfromj = {}                             #reaches downstream of junction

junc_res_sink_next_junc_down = {}       #junctions reservoirs and sinks and their next downstream junction (strict)
dsjunction_next_junc_down = {}          #downstream junctions and the next downstream junction (strict)

In [32]:
#will look through the gdfs provided and collect the source of each for this file. The source should be the HUC8 of the HMS model the schematic file came from.
#This will create the hms_origin_dictionary
initial_list_2 = [junctions, reaches, subbasins, sinks, sources, reservoirs]
edited_list_2 = [i for i in initial_list_2 if len(i) != 0]

hms_origin_dictionary = single_dict_creator(gdf_list=edited_list_2,identify_field='name',connection_field='Source_Path',supplemental_dicts=[])

In [ ]:
#The following dictionaries will involve categorizing the source data based on how it should be incorporated into the model.
source_ds_to = sources.set_index('name').to_dict()['Downstream']
print(sources['name'].to_list())

In [ ]:
#Determine if a source should be considered incrementally. This means it's flow does not replace, but is added to the flow of the junction it ties into.
userinput4 = input(f'Review the sources. Type list of the sources that will be considered reservoirs in terms of model incorporation (IF CUMMULATIVE). Make it a list format ex. [\'res_1\',\'res_2\']\nNote: If listed, these are nodes whose flow will be directly incorporated into the model as a node. Otherwise the sources will not be used.')
res_from_so = ast.literal_eval(userinput4)

for res_inc in res_from_so:
    source_if_inc[res_inc] = source_ds_to[res_inc]

In [ ]:
#Dictionaries related to the sources file if NOT to be used incrementally along with subbasins.
userinput5 = input(f'Review the sources. Type list of the sources that will be considered reservoirs in terms of model incorporation (IF CUMMULATIVE). Make it a list format ex. [\'res_1\',\'res_2\']\nNote: If listed, these are nodes whose flow will be directly incorporated into the model as a node. Otherwise the sources will not be used.')
res_from_so = ast.literal_eval(userinput5)

for source in sources['name'].to_list():
    if source not in res_from_so:
        source_if_res[source] = 'FALSE'
    else:
        source_text = f'\033[4m{source}\033[0m\033[31m'
        text = f'Please review on arcgis whether the coordinates of {source_text} node will need to be changed for introduction of flow.\nIf the answer is yes, please type the new coordinates in the following format such that:\n\nThe coordinate units match the projection of the shapefile (in US-Ft)\nThe new coordinates are south of the dam crest and model "Notch" at the reservoir (if in the middle of the model) \nEast and North are positive\nWest and South are negative \nEx. [-3780441.02,7215490.00] if the coordinate are 3780441.02W and 7215490.00N.\nIf there is no coordinate change required, leave the brackets empty as []' 
        print(f"\033[31m{text}\033[0m")
        userinput5 = input(f'Type response:')
        coord_change = ast.literal_eval(userinput5)
        if coord_change == []:
            source_if_res[source] = {'TRUE':'None'}
        else:
            source_if_res[source] = {'TRUE':coord_change}

['HUC_001_Source_002', 'Source_003_to_J228', 'Pilot_Butte_Res2', 'Pilot_Butte_Res1', 'Pilot_Butte_Res3', 'Pilot_Butte_Res4', 'HUC_005_Source_001', 'HUC_005_Source_004', 'HUC_005_Source_006', 'HUC_005_Source_06259000', 'HUC_007_Source_005', 'HUC_007_Source_008', 'HUC_010_Source_007', 'HUC_010_Source_009', 'HUC_010_Source_011', 'HUC_010_Source_014', 'HUC_012_Source_013', 'HUC_014_Source_012']
Review the souces. Type list of the sources that will be considered reservoirs in terms of model incorporation. Make it a list format ex. ['res_1','res_2'] ["N\A"]
Please review what the nearest sink to source feature HUC_001_Source_002 is as a list. Ex. ["Sink1234"] ["N\A"]
Please review what the nearest sink to source feature Source_003_to_J228 is as a list. Ex. ["Sink1234"] ["N\A"]
Please review what the nearest sink to source feature Pilot_Butte_Res2 is as a list. Ex. ["Sink1234"] ["N\A"]
Please review what the nearest sink to source feature Pilot_Butte_Res1 is as a list. Ex. ["Sink1234"] ["N\A"

In [35]:
#we have the node_to and reach_to dictionaries however they do not filter specifically to junctions and reaches. They will be filtered in search of only junctions and only reaches.
jtor = {k:v for k,v in node_to.items() if v in reaches['name'].to_list()}
rtoj = {i:j for i,j in reach_to.items()}
rfromj = {b:a for a,b in node_to.items() if b in reaches['name'].to_list()}

#a dictionary exists with junctions reservoirs sinks and sources - to create junc_res_sink_next_junc_down we need to create it 
junc_res_sink_next_junc_down = {key:val for key,val in node_to.items() if key not in sources['name'].to_list()}
for key,val in junc_res_sink_next_junc_down.items():
    identifiedj = [x for x in [val] if x in junctions['name'].to_list()]
    if val == 'OUT':
        identifiedj = ['OUT']
    if val == 'N\A':
        identifiedj = ['N\A']
    while len(identifiedj) == 0:
        nextj = singular_network([val],connection_dictionary,restriction_str='_R_')
        if nextj == [] or nextj == ['OUT'] or nextj == ['missing connection'] or nextj == ['N\A']:
            identifiedj = ['']
            if nextj == ['OUT']:
                identifiedj = ['OUT']
            if nextj == ['N\A']:
                identifiedj = ['N\A']
        else:
            identifiedj = [x for x in nextj if x in junctions['name'].to_list()]
            val = nextj[0]
    
    update_dict = {key:identifiedj[0]}
    if identifiedj[0] not in junctions['name'].to_list():
        print('Exceptions within junc_res_sink_next_junc_down dictionary: ',update_dict)
    junc_res_sink_next_junc_down.update(update_dict)

dsjunction_next_junc_down = {key:val for key,val in junc_res_sink_next_junc_down.items() if key in ds_junc_to_HUC10.keys()}
    
print('Final dictionaries created.')

Exceptions within junc_res_sink_next_junc_down dictionary:  {'HUC_010_Sink_0': 'OUT'}
Exceptions within junc_res_sink_next_junc_down dictionary:  {'HUC_016_Sink_0': 'OUT'}
Exceptions within junc_res_sink_next_junc_down dictionary:  {'HUC_010_J_327': 'OUT'}
Exceptions within junc_res_sink_next_junc_down dictionary:  {'HUC_016_J_1': 'OUT'}
Final dictionaries created.


In [36]:
len(dsjunction_next_junc_down)

78

## Export all Dictionaries as JSONs - Remove those which are no longer needed

In [ ]:
#each of the dictionaries is exported as a json file

## Group 1 of dictionaries
with open(outputs/"HUC10_Junctions_Subbasins.json","w") as outfile:
    json.dump(HUC10_Junc_Sub,outfile, indent = 2)
with open(outputs/"HUC10_Junctions.json","w") as outfile2:
    json.dump(HUC10_Junc,outfile2, indent = 2)
with open(outputs/"HUC10_Subbasins.json","w") as outfile3:
    json.dump(HUC10_Sub,outfile3, indent = 2)
with open(outputs/"Junction_Subbasins.json","w") as outfile4:
    json.dump(Junc_Sub,outfile4, indent = 2)

## Group 2 of dictionaries
with open(outputs/"HUC10_into_dsJunction.json","w") as outfile5:
    json.dump(HUC10_to_ds_junc,outfile5, indent = 2)
with open(outputs/"dsJunction_out_toHUC10.json","w") as outfile6:
    json.dump(ds_junc_to_HUC10,outfile6, indent = 2)
with open(outputs/"inHUC10_dsJunction_outHUC10.json","w") as outfile7:
    json.dump(HUC10_dsjunc_HUC10,outfile7, indent = 2)
with open(outputs/"HUC10_outflow_toHUC10.json","w") as outfile8:
    json.dump(HUC10_to_HUC10,outfile8, indent = 2)
with open(outputs/"outofscope_HUC10_to_HUC10.json","w") as outfile9:
    json.dump(HUC10_to_HUC10_outofscope,outfile9, indent = 2)

## Group 3 of dictionaries

with open(outputs/"source_to_downstream.json","w") as outfile11:
    json.dump(source_ds_to,outfile11, indent = 2)
with open(outputs/"source_if_reservoir.json","w") as outfile12:
    json.dump(source_if_res,outfile12, indent = 2)
with open(outputs/"source_if_incremental.json","w") as outfile13:
    json.dump(source_if_inc,outfile13, indent = 2)


with open(outputs/"hms_feature_origin_dict.json","w") as outfile14:
    json.dump(hms_origin_dictionary,outfile14, indent = 2)

with open(outputs/"reachtojunction.json","w") as outfile15:
    json.dump(rtoj,outfile15, indent = 2)
with open(outputs/"reachfromjunction.json","w") as outfile16:
    json.dump(rfromj,outfile16, indent = 2)
with open(outputs/"junctiontoreach.json","w") as outfile17:
    json.dump(jtor,outfile17, indent = 2)

with open(outputs/"junc_res_sink_next_junc_down.json","w") as outfile18:
    json.dump(junc_res_sink_next_junc_down,outfile18, indent = 2)
with open(outputs/"dsjunction_next_junc_down.json","w") as outfile19:
    json.dump(dsjunction_next_junc_down,outfile19, indent = 2)

In [38]:
#The following is a dictionary with an explanation of each dictionary above and additional details for each for review.
delimeter = '\n'
current_date = datetime.date.today()
string_list = [f'{current_date} - Dictionaries from Final_SST schematics',
                             '\ndsHUC10_Junctions_Subbasins.json:',
                             f'\t A HUC10, the Junctions and the Subbasins that flow into them. Total HUC10s: {len(HUC10_Junc_Sub.keys())}',
                             '\nHUC10_Junctions.json:',
                             '\t A HUC10 and its assigned junctions',
                             '\nHUC10_Subbasins.json:',
                             '\t A HUC10 and its assigned subbasins - the ones the model domain is based on',
                             '\nJunction_Subbasins.json:',
                             '\t Junction and the Subbasins that flow into that Junction',
                             '\nHuc10_into_dsJunction.json:',
                             '\t HUC10 and their assigned downstream junction - This is the junction at the downstream boundary condition of the model and will be applied as inflow to the next model',
                             '\ndsJunction_out_toHUC10.json:',
                             '\t Each downstream junction and the HUC10 that it flow INTO',
                             '\ninHUC10_dsJunction_outHUC10.json:',
                             '\t A HUC10, its downstream junction and the HUC10 it flows INTO',
                             '\nHUC10_outflow_toHUC10.json:',
                             '\t Scoped HUC10 models and the HUC10s they flow into',
                             '\noutofscope_HUC10_to_HUC10.json:',
                             '\t Unscoped HUC10 models and the HUC10s they flow into',
                             '\n------------SCRAPPED'
                             '\nsource_to_nearest_sink.json:',
                             '\t source feature from source shapefile and the nearest sink',
                             '\n--------------------'
                             '\nsource_to_downstream.json:',
                             '\t source feature and the downstream feature (based on the downstream field in its attribute table.)',
                             '\nsource_if_reservoir.json:',
                             '\t True or False if a source feature from the shapefile will be incorporated as a reservoir. if True, there may be a coordinate input after for where to apply the downstream flow',
                             '\nhms_feature_origin_dict.json:',
                             '\t each feature and the source HMS model it comes from',
                             '\nreachtojunction.json:',
                             '\t reach to its downstream based on the attribute table',
                             '\nreachfromjunction.json:',
                             '\t reach and the junction it receives flow FROM',
                             '\njunctiontoreach.json:',
                             '\t junction and what reach it flows INTO',
                             '\njunc_res_sink_next_junc_down.json:',
                             '\t junctions, reservoirs, sinks and their nearest downstream junction.',
                             '\ndsjunction_next_junc_down.json:',
                             '\t downstream junctions and the nearest downstream junction from them.']

explain_dict = delimeter.join(string_list)
text_file = open(outputs/"dictionary_explanations.txt","w")
text_file.write(explain_dict)
text_file.close()

In [39]:
# print(explain_dict)

### Final Notes

The dictionaries and the order in which they are identified

HUC10_Junc_Sub        
HUC10_Junc         
HUC10_Sub              
Junc_Sub  

HUC10_to_ds_junc            
ds_junc_to_HUC10           
HUC10_dsjunc_HUC10          
HUC10_to_HUC10              
HUC10_to_HUC10_outofscope
                 
source_ds_to                    
source_if_res   

hms_origin_dictionary       

rtoj                         
rfromj
jtor

junc_res_sink_next_junc_down 
dsjunction_next_junc_down

A dictionary that will need to be created manually using models, available DEM, and aerial imagery. There is a notebook that exports a json from an excel table
rating_or_stage

Finally, below are dictionaries that will be made in later steps (not this notebook)
event_dictionary_creation
sequencer

In [120]:
#Complete